### Why this notebook uses the Azure SDK instead of a mounted filesystem

The original tutorial mounts ADLS Gen2 using an app registration (client ID,
tenant ID, client secret). That approach depends on `dbutils.fs.mount`,
which is **not supported on Serverless compute** (Spark Connect restricts
direct Hadoop filesystem configuration).

Serverless was used here because UK South had no available cluster
capacity on this trial subscription at the time of building — see the
README for the full regional-capacity story. As a workaround, this
notebook connects directly to ADLS Gen2 using the `azure-storage-blob` SDK,
reading and writing blobs in-memory via pandas rather than mounting a
filesystem path.

In [0]:
%pip install azure-storage-blob

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Installing `azure-storage-blob` via `%pip install` only takes effect after
the Python kernel restarts, so the next cell restarts it before the SDK is
imported and used.

In [0]:
dbutils.library.restartPython()

**Security note:** This notebook authenticates using the storage account's
**access key**, which grants full read/write access to every container in
the account. A scoped service principal (app registration) would be the
more secure option in production, but it's unsupported on Serverless
compute for the reason above. The access key below is a placeholder,
replace it with your own storage account key

In [0]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import StringIO

storage_account_name = "tokyoolympicdatasheikh"
storage_account_key = "YOUR_STORAGE_ACCOUNT_KEY_HERE"  # from Azure Portal: Storage account > Access keys

connection_string = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_name = "tokyo-olympic-data"
blob_name = "raw-data/Athletes.csv"

blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
blob_data = blob_client.download_blob().readall().decode('utf-8', errors='replace')

pdf = pd.read_csv(StringIO(blob_data))
df_athletes = spark.createDataFrame(pdf)
df_athletes.show(5)

+-----------------+-------+-------------------+
|       PersonName|Country|         Discipline|
+-----------------+-------+-------------------+
|  AALERUD Katrine| Norway|       Cycling Road|
|      ABAD Nestor|  Spain|Artistic Gymnastics|
|ABAGNALE Giovanni|  Italy|             Rowing|
|   ABALDE Alberto|  Spain|         Basketball|
|    ABALDE Tamara|  Spain|         Basketball|
+-----------------+-------+-------------------+
only showing top 5 rows


### Loading all datasets via a reusable function

Rather than repeating the download-decode-read logic for every CSV,
`load_csv_as_spark_df()` wraps it into a single reusable function: given a
container and blob path, it downloads the blob, decodes it, reads it into
pandas, and returns it as a Spark DataFrame. Each of the five raw datasets
is then loaded with one call.

In [0]:
def load_csv_as_spark_df(container_name, blob_path):
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_path)
    blob_data = blob_client.download_blob().readall().decode('latin-1')
    pdf = pd.read_csv(StringIO(blob_data))
    return spark.createDataFrame(pdf)

df_coaches = load_csv_as_spark_df("tokyo-olympic-data", "raw-data/Coaches.csv")
df_entries_gender = load_csv_as_spark_df("tokyo-olympic-data", "raw-data/EntriesGender.csv")
df_medals = load_csv_as_spark_df("tokyo-olympic-data", "raw-data/Medals.csv")
df_teams = load_csv_as_spark_df("tokyo-olympic-data", "raw-data/Teams.csv")

df_coaches.show(5)
df_entries_gender.show(5)
df_medals.show(5)
df_teams.show(5)

+---------------+-------------+----------+-----+
|           Name|      Country|Discipline|Event|
+---------------+-------------+----------+-----+
|ABDELMAGID Wael|        Egypt|  Football| NULL|
|      ABE Junya|        Japan|Volleyball| NULL|
|  ABE Katsuhiko|        Japan|Basketball| NULL|
|   ADAMA Cherif|Côte d'Ivoire|  Football| NULL|
|     AGEBA Yuya|        Japan|Volleyball| NULL|
+---------------+-------------+----------+-----+
only showing top 5 rows
+-------------------+------+----+-----+
|         Discipline|Female|Male|Total|
+-------------------+------+----+-----+
|     3x3 Basketball|    32|  32|   64|
|            Archery|    64|  64|  128|
|Artistic Gymnastics|    98|  98|  196|
|  Artistic Swimming|   105|   0|  105|
|          Athletics|   969|1072| 2041|
+-------------------+------+----+-----+
only showing top 5 rows
+----+--------------------+----+------+------+-----+-------------+
|Rank|         TeamCountry|Gold|Silver|Bronze|Total|Rank by Total|
+----+-----------

### Inferring and Casting Schema Data Types

Columns read in from CSV default to string type. Before running any
numeric aggregation (totals, ratios, counts), the relevant columns are
cast to integer type using a reusable `cast_columns_to_integer()` helper,
so the same logic can be applied across all five datasets without
repeating the casting code per column.


In [0]:
df_athletes.printSchema()
df_coaches.printSchema()

root
 |-- PersonName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)

root
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Event: string (nullable = true)



In [0]:
df_entries_gender.printSchema()

root
 |-- Discipline: string (nullable = true)
 |-- Female: long (nullable = true)
 |-- Male: long (nullable = true)
 |-- Total: long (nullable = true)



### Casting numeric columns to integer type

`Female`, `Male`, and `Total` in the entries dataset are read in as
strings and need to be cast to integers before any aggregation. This is
done with `withColumn()` and `.cast(IntegerType())` per column.

The medals dataset has several numeric columns to cast the same way, so
the next section applies this via the reusable `cast_columns_to_integer()`
helper instead of repeating `.withColumn()` calls one by one.

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DataType


df_entries_gender = df_entries_gender.withColumn("Female",col("Female").cast(IntegerType()))\
    .withColumn("Male", col("Male").cast(IntegerType()))\
    .withColumn("Total", col("Total").cast(IntegerType()))

In [0]:
df_entries_gender.printSchema()

root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)



### Casting the medals columns using a reusable function

The medals dataset has six numeric columns to cast, rather than three like
the gender dataset, so the repeated `.withColumn()` chain used above would
get long and repetitive. Instead, `cast_columns_to_integer()` takes a
DataFrame and a list of column names and casts each one to integer in a
loop, scaling to any number of columns without extra code per column.

In [0]:
df_medals.printSchema()

root
 |-- Rank: long (nullable = true)
 |-- TeamCountry: string (nullable = true)
 |-- Gold: long (nullable = true)
 |-- Silver: long (nullable = true)
 |-- Bronze: long (nullable = true)
 |-- Total: long (nullable = true)
 |-- Rank by Total: long (nullable = true)



In [0]:
def cast_columns_to_integer(df, column_names):
    for column_name in column_names:
        df = df.withColumn(column_name, col(column_name).cast(IntegerType()))
    return df

df_medals = cast_columns_to_integer(df_medals, ["Rank", "Gold", "Silver", "Bronze", "Total", "Rank by Total"])
df_medals.printSchema()

root
 |-- Rank: integer (nullable = true)
 |-- TeamCountry: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank by Total: integer (nullable = true)



In [0]:
df_teams.printSchema()

root
 |-- TeamName: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Event: string (nullable = true)



### Baseline analytics on the transformed data

Before building the two original metrics (medal efficiency and gender
parity), a few baseline queries are run directly on the transformed Spark
DataFrames — top medal-winning countries, gender splits, etc. These mirror
the tutorial's original analysis and confirm the type-cast data behaves as
expected ahead of the custom metrics.

In [0]:
# Find the Top countries with the highest number of Gold Medals
top_gold_medal_countries = df_medals.orderBy("Gold", ascending=False).select("TeamCountry","Gold").show()

+--------------------+----+
|         TeamCountry|Gold|
+--------------------+----+
|United States of ...|  39|
|People's Republic...|  38|
|               Japan|  27|
|       Great Britain|  22|
|                 ROC|  20|
|           Australia|  17|
|             Germany|  10|
|               Italy|  10|
|         Netherlands|  10|
|              France|  10|
|         New Zealand|   7|
|              Canada|   7|
|                Cuba|   7|
|              Brazil|   7|
|             Hungary|   6|
|   Republic of Korea|   6|
|      Czech Republic|   4|
|               Kenya|   4|
|              Poland|   4|
|              Norway|   4|
+--------------------+----+
only showing top 20 rows


In [0]:
#Calculate the Avergae number of entries by gender for each discipline
from pyspark.sql.functions import round as spark_round

def add_ratio_column(df, numerator_col, denominator_col, new_col_name, decimals=3):
    return df.withColumn(
        new_col_name,
        spark_round(col(numerator_col) / col(denominator_col), decimals)
    )

average_entries_by_gender = add_ratio_column(df_entries_gender, "Female", "Total", "Avg_Female")
average_entries_by_gender = add_ratio_column(average_entries_by_gender, "Male", "Total", "Avg_Male")

average_entries_by_gender.show()

+--------------------+------+----+-----+----------+--------+
|          Discipline|Female|Male|Total|Avg_Female|Avg_Male|
+--------------------+------+----+-----+----------+--------+
|      3x3 Basketball|    32|  32|   64|       0.5|     0.5|
|             Archery|    64|  64|  128|       0.5|     0.5|
| Artistic Gymnastics|    98|  98|  196|       0.5|     0.5|
|   Artistic Swimming|   105|   0|  105|       1.0|     0.0|
|           Athletics|   969|1072| 2041|     0.475|   0.525|
|           Badminton|    86|  87|  173|     0.497|   0.503|
|   Baseball/Softball|    90| 144|  234|     0.385|   0.615|
|          Basketball|   144| 144|  288|       0.5|     0.5|
|    Beach Volleyball|    48|  48|   96|       0.5|     0.5|
|              Boxing|   102| 187|  289|     0.353|   0.647|
|        Canoe Slalom|    41|  41|   82|       0.5|     0.5|
|        Canoe Sprint|   123| 126|  249|     0.494|   0.506|
|Cycling BMX Frees...|    10|   9|   19|     0.526|   0.474|
|  Cycling BMX Racing|  

## Custom Analytical Facts

The base pipeline (tutorial) computes two baseline queries — top gold medal countries, and average entries by gender — but only displays them; they are never persisted.

This section extends that with two original derived metrics, computed and **written to the transformed layer as their own datasets**, not just printed:

1. **Medals-per-athlete (efficiency)** — normalizes medal count by squad size, surfacing which countries performed best *relative to the number of athletes they sent*, rather than just who has the biggest team.
2. **Gender Parity Index** — a single comparable score (0 to 1) measuring how close each discipline is to perfect gender balance, rather than just raw Female/Male ratios.

In [0]:
from pyspark.sql.functions import count, col, round as spark_round, abs as spark_abs

# ----- Metric 1: Medals-per-athlete (efficiency) -----
athlete_counts = df_athletes.groupBy("Country").agg(count("PersonName").alias("AthleteCount"))

medal_efficiency = df_medals.join(
    athlete_counts,
    df_medals.TeamCountry == athlete_counts.Country,
    "inner"
).withColumn(
    "MedalsPerAthlete",
    spark_round(col("Total") / col("AthleteCount"), 3)
).select("TeamCountry", "Total", "AthleteCount", "MedalsPerAthlete") \
 .orderBy(col("MedalsPerAthlete").desc())

medal_efficiency.show(10)

+--------------------+-----+------------+----------------+
|         TeamCountry|Total|AthleteCount|MedalsPerAthlete|
+--------------------+-----+------------+----------------+
|          San Marino|    3|           4|            0.75|
|             Bermuda|    1|           2|             0.5|
|             Armenia|    4|          16|            0.25|
|             Georgia|    8|          35|           0.229|
|                 ROC|   71|         318|           0.223|
|         Philippines|    4|          18|           0.222|
|People's Republic...|   88|         401|           0.219|
|                Cuba|   15|          69|           0.217|
|               Qatar|    3|          14|           0.214|
|              Kosovo|    2|          10|             0.2|
+--------------------+-----+------------+----------------+
only showing top 10 rows


Countries with small squads but strong per-athlete returns appear at the top here , a very different leaderboard than the raw gold-medal count, which simply rewards the largest teams.

In [0]:
# ----- Metric 2: Gender Parity Index -----

def add_ratio_column(df, numerator_col, denominator_col, new_col_name, decimals=3):
    return df.withColumn(new_col_name, spark_round(col(numerator_col) / col(denominator_col), decimals))

gender_parity = df_entries_gender.withColumn("AbsDifference", spark_abs(col("Female") - col("Male")))
gender_parity = add_ratio_column(gender_parity, "AbsDifference", "Total", "ImbalanceRatio")
gender_parity = gender_parity.withColumn("ParityIndex", spark_round(1 - col("ImbalanceRatio"), 3))

gender_parity = gender_parity.select("Discipline", "Female", "Male", "Total", "ParityIndex") \
    .orderBy(col("ParityIndex").desc())

gender_parity.show(10)

+--------------------+------+----+-----+-----------+
|          Discipline|Female|Male|Total|ParityIndex|
+--------------------+------+----+-----+-----------+
|                Golf|    60|  60|  120|        1.0|
|Cycling Mountain ...|    38|  38|   76|        1.0|
|    Beach Volleyball|    48|  48|   96|        1.0|
|            Handball|   168| 168|  336|        1.0|
|             Archery|    64|  64|  128|        1.0|
|        Canoe Slalom|    41|  41|   82|        1.0|
|      3x3 Basketball|    32|  32|   64|        1.0|
|          Basketball|   144| 144|  288|        1.0|
| Artistic Gymnastics|    98|  98|  196|        1.0|
|  Cycling BMX Racing|    24|  24|   48|        1.0|
+--------------------+------+----+-----+-----------+
only showing top 10 rows


A ParityIndex of **1.0** means perfectly balanced participation; lower scores flag disciplines still skewed toward one gender — a more directly actionable signal than the raw ratio columns alone.

## Persisting Derived Facts

Unlike the base tutorial, which stops at printing these aggregates, both derived metrics are written to the transformed-data layer as their own datasets,treating them as queryable fact tables rather than throwaway output.

In [0]:
def write_spark_df_as_csv(spark_df, container_name, blob_path):
    pdf = spark_df.toPandas()
    csv_buffer = StringIO()
    pdf.to_csv(csv_buffer, index=False)

    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_path)
    blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)

# Base datasets
write_spark_df_as_csv(df_coaches, "tokyo-olympic-data", "transformed-data/coaches.csv")
write_spark_df_as_csv(df_entries_gender, "tokyo-olympic-data", "transformed-data/entriesgender.csv")
write_spark_df_as_csv(df_medals, "tokyo-olympic-data", "transformed-data/medals.csv")
write_spark_df_as_csv(df_teams, "tokyo-olympic-data", "transformed-data/teams.csv")
write_spark_df_as_csv(df_athletes, "tokyo-olympic-data", "transformed-data/athletes.csv")

# Derived facts
write_spark_df_as_csv(medal_efficiency, "tokyo-olympic-data", "transformed-data/medal_efficiency.csv")
write_spark_df_as_csv(gender_parity, "tokyo-olympic-data", "transformed-data/gender_parity_index.csv")